# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
from src.provenance import verify_checkout, verify_meridian_origin
print("Código:",verify_checkout(PROJECT_ROOT, REPO_REF))
print("Meridian:",verify_meridian_origin())
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Código: {'requested_ref': 'main', 'resolved_commit': 'ac9960a3a86e7af07a507121043c6914fcbc4195', 'dirty': True}
Meridian: {'url': 'https://github.com/google/meridian.git', 'vcs_info': {'commit_id': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'requested_revision': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'vcs': 'git'}}
Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-revision; painel: (6240, 19)


O manifesto verifica código, raw, ambiente, conclusão das etapas e hashes dos outputs; resultados incompatíveis são recalculados.

In [2]:
from src.provenance import require_stages
try:
    require_stages(range(4))
except RuntimeError as stale:
    print('Recalculando etapas anteriores:',stale)
    from src.eda import data_audit,general,temporal,relationships
    from src.geo_analysis import geo_analysis,media_geo
    from src.official_eda import official_eda
    for analysis in [data_audit,general,temporal,geo_analysis,media_geo,relationships,official_eda][:4]:
        analysis(geo_time_panel)


## 04 — Como e onde os canais recebem investimento?

### Pergunta
Como e onde os canais recebem investimento?

### Motivação
O valor do painel para MMM depende também de execução territorial e temporal diferenciada, e não somente do número de geos.

### Método
Matrizes de gasto, mix dentro de cada geo e alocação de cada canal; intensidade por habitante; Spearman/Pearson com população; HHI; CV within; heatmaps GEO × semana para gasto e impressões, brutos e per capita.

### Interpretação antes de olhar os resultados
Mix tem denominador por geo; alocação tem denominador por canal. HHI mede concentração, não performance. Diferenças de cor entre figuras não são comparáveis sem observar escalas. Geos seguem ordem fixa pelo KPI total.

**Dependências:** painel validado e funções em `src/geo_analysis.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [3]:
from src.geo_analysis import media_geo
report_04 = media_geo(geo_time_panel)
show(report_04,part="summary")

### Resultado observado e conclusão parcial

O mix divide cada linha GEO × canal pelo orçamento total do geo; a alocação divide cada coluna pelo orçamento do canal. Essas perguntas são diferentes. A maior amplitude de share do mix é 3.74 pontos percentuais, em Channel1.

HHI é a soma dos quadrados das participações geográficas de um canal: varia de 1/G (uniforme) a 1 (um único geo). Observamos 0.030 a 0.031, sem aplicar limiares de concorrência econômica. O inverso traduz concentração em número equivalente de geos uniformes, não em tamanho amostral efetivo.

Os CVs within variam de 0.442 a 2.153. Médias zero produzem CV indefinido. Heatmaps usam a mesma ordenação de geos por KPI total e escalas de cor próprias por variável; não comparar cores entre painéis sem ler as escalas. As versões por habitante ajudam a investigar diferenças que persistem além do tamanho dos mercados.

**Mix relativamente semelhante:** dispersões são expressas em pontos percentuais; o gráfico de desvios do nacional expõe pequenas diferenças sem sugerir forte heterogeneidade acumulada.

Channel0: mix 16.70%–20.00%, amplitude 3.30 p.p., desvio 0.74 p.p.; Spearman população×volume 0.993, população×intensidade 0.286; R² GEO bruto/per capita 0.248/0.006; resíduo per capita 0.783.

Channel1: mix 12.36%–16.10%, amplitude 3.74 p.p., desvio 0.82 p.p.; Spearman população×volume 0.987, população×intensidade 0.047; R² GEO bruto/per capita 0.144/0.004; resíduo per capita 0.765.

Channel2: mix 4.04%–6.46%, amplitude 2.42 p.p., desvio 0.57 p.p.; Spearman população×volume 0.966, população×intensidade 0.206; R² GEO bruto/per capita 0.053/0.004; resíduo per capita 0.793.

Channel3: mix 38.00%–41.36%, amplitude 3.36 p.p., desvio 0.91 p.p.; Spearman população×volume 0.994, população×intensidade 0.142; R² GEO bruto/per capita 0.390/0.005; resíduo per capita 0.648.

Channel4: mix 20.56%–23.92%, amplitude 3.37 p.p., desvio 0.79 p.p.; Spearman população×volume 0.993, população×intensidade 0.232; R² GEO bruto/per capita 0.237/0.004; resíduo per capita 0.706.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** **Mix relativamente semelhante:** dispersões são expressas em pontos percentuais; o gráfico de desvios do nacional expõe pequenas diferenças sem sugerir forte heterogeneidade acumulada.
- **Quais problemas apareceram?** Resíduo pode conter alternância de atividade e ruído, não apenas sinal causal.
- **O que falta investigar?** Correlações entre canais e controles, VIF e conflitos com geo/tempo.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


### Evidências e leitura das figuras
Tabelas pequenas são exibidas integralmente. As sínteses selecionam os casos relevantes; os CSVs mantêm os dados completos.

In [4]:
show(report_04,part="evidence")

**media_mix** — 40 linhas; CSV completo em `outputs/tables/`.

channel,Channel0,Channel1,Channel2,Channel3,Channel4
geo,,,,,
Geo36,0.190032,0.145227,0.061584,0.389280,0.213879
Geo39,0.198288,0.138034,0.048407,0.387838,0.227433
Geo10,0.177915,0.137524,0.049388,0.411376,0.223797
Geo23,0.184789,0.147995,0.059540,0.401638,0.206038
Geo31,0.199994,0.154922,0.050278,0.380017,0.214790
Geo7,0.189596,0.135009,0.060125,0.393955,0.221316
Geo2,0.179638,0.139828,0.061873,0.406928,0.211733
Geo25,0.192168,0.149787,0.049639,0.394906,0.213501
Geo17,0.195987,0.152988,0.046717,0.389405,0.214902


**geo_allocation** — 40 linhas; CSV completo em `outputs/tables/`.

channel,Channel0,Channel1,Channel2,Channel3,Channel4
geo,,,,,
Geo36,0.045798,0.045260,0.049764,0.043248,0.043739
Geo39,0.038196,0.034384,0.031265,0.034439,0.037175
Geo10,0.042522,0.042504,0.039579,0.045324,0.045387
Geo23,0.045338,0.046955,0.048981,0.045426,0.042895
Geo31,0.042802,0.042875,0.036079,0.037491,0.039006
Geo7,0.036130,0.033270,0.038417,0.034607,0.035787
Geo2,0.041837,0.042112,0.048317,0.043688,0.041844
Geo25,0.037912,0.038214,0.032836,0.035915,0.035742
Geo17,0.038819,0.039185,0.031025,0.035554,0.036118


**concentration** — 5 linhas; CSV completo em `outputs/tables/`.

...

[Figura gerada; veja outputs/figures ou execute a célula.]

**Como ler — within cv distribution:** Compare com os resultados numéricos e a interpretação do capítulo. Unidade e denominador delimitam a leitura; associação não demonstra eficácia.

[Figura gerada; veja outputs/figures ou execute a célula.]

**Como ler — mix deviations pp:** Diferenças em pontos percentuais; mix acumulado semelhante pode coexistir com execução semanal diferente.

[Figura gerada; veja outputs/figures ou execute a célula.]